In [2]:
%%writefile /content/pdf_web_crawler.py
#!/usr/bin/env python3
"""
pdf_web_crawler.py
==================
Crawls seed websites → discovers ALL PDFs → downloads them →
rebuilds FAISS + BM25 indexes.

Drive folders:
  FranchiseOps → MyDrive/FranchiseOps_AI/rag_pdfs/
  FreightQuote → MyDrive/FreightQuote_AI/rag_pdfs/
"""

import os
import re
import json
import time
import pickle
import hashlib
import requests
import urllib.parse
import subprocess
import sys

from pathlib import Path
from collections import deque
from datetime import datetime

# Dependencies are installed by the notebook setup cell before this script runs.

from bs4 import BeautifulSoup
from tqdm.auto import tqdm

# ─── Drive folder names ───────────────────────────────────────────────────────
FC_DRIVE_FOLDER = "FranchiseOps_AI"


# ─── Crawler settings ─────────────────────────────────────────────────────────
HEADERS = {
    "User-Agent": ("Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
                   "Chrome/120.0.0.0 Safari/537.36"),
    "Accept": "text/html,application/xhtml+xml,*/*;q=0.8",
}
PDF_HEADERS = {
    "User-Agent": HEADERS["User-Agent"],
    "Accept": "application/pdf,*/*",
}

# Milestone 3 Task 1 collection settings.
MAX_CRAWL_DEPTH   = 3
MAX_PDFS_PER_SEED = 60
MAX_CRAWL_PAGES   = 1200   # global unique HTML-page budget across all seed URLs
TARGET_PDF_COUNT  = 150
MAX_PDF_BYTES     = 50 * 1024 * 1024
DOWNLOAD_TIMEOUT  = 40
REQUEST_TIMEOUT   = 20
DELAY_BETWEEN     = 0.35
CHUNK_WORDS       = 400
CHUNK_OVERLAP     = 50

# ─────────────────────────────────────────────────────────────────────────────
# SEED URLS
# ─────────────────────────────────────────────────────────────────────────────

FRANCHISE_SEEDS = [
    ("Web Source", "https://www.osha.gov/bloodborne-pathogens"),
    ("Web Source", "https://www.fda.gov/food/retail-food-protection"),
    ("Web Source", "https://www.fssai.gov.in/cms/food-safety-and-standards-act-2006.php"),
    ("Web Source", "https://www.fssai.gov.in/cms/draft-notifications.php"),
    ("Web Source", "https://www.mofpi.gov.in/"),
    ("Web Source", "https://www.servsafe.com/food-manager"),
    ("Web Source", "https://www.cdc.gov/foodsafety/index.html"),
    ("Web Source", "https://www.who.int/news-room/fact-sheets/detail/food-safety"),
    ("Web Source", "https://www.fssai.gov.in/cms/gazette-notifications.php"),
    ("Web Source", "https://www.fssai.gov.in/cms/manuals-of-methods-of-analysis-of-various-food-products.php"),
    ("Web Source", "https://labour.gov.in/maternity-benefit-act"),
    ("Web Source", "https://www.osha.gov/restaurants"),
    ("Web Source", "https://www.mygfsi.com/"),
    ("Web Source", "https://www.franchise.org/franchise-information/franchise-legal"),
    ("Web Source", "https://www.who.int/health-topics/food-safety"),
    ("Web Source", "https://www.fda.gov/food/guidance-regulation-food-and-dietary-supplements"),
    ("Web Source", "https://www.franchise.org/franchise-information/franchise-finance"),
    ("Web Source", "https://www.cdc.gov/foodsafety/outbreaks/index.html"),
    ("Web Source", "https://labour.gov.in/employees-provident-fund-organisation"),
    ("Web Source", "https://www.fda.gov/food/hazard-analysis-critical-control-point-haccp"),
    ("Web Source", "https://www.osha.gov/emergency-preparedness"),
    ("Web Source", "https://www.osha.gov/personal-protective-equipment"),
    ("Web Source", "https://www.franchise.org/franchise-information/franchise-business-opportunities"),
    ("Web Source", "https://www.franchise.org/franchise-opportunities/retail"),
    ("Web Source", "https://www.fda.gov/food/foodborne-pathogens"),
    ("Web Source", "https://labour.gov.in/social-security"),
    ("Web Source", "https://www.nraef.org/servsafe/"),
    ("Web Source", "https://www.fda.gov/food/food-safety-modernization-act-fsma"),
    ("Web Source", "https://www.fda.gov/food/food-safety-during-emergencies"),
    ("Web Source", "https://www.brcgs.com/our-standards/food-safety/"),
    ("Web Source", "https://www.tea.in/"),
    ("Web Source", "https://www.fssai.gov.in/cms/notices.php"),
    ("Web Source", "https://labour.gov.in/industrial-relations"),
    ("Web Source", "https://www.franchise.org/franchise-information/franchise-management"),
    ("Web Source", "https://labour.gov.in/employees-state-insurance-corporation"),
    ("Web Source", "https://www.franchise.org/franchise-information/franchise-marketing"),
    ("Web Source", "https://consumeraffairs.nic.in/"),
    ("Web Source", "https://dca.nic.in/"),
    ("Web Source", "https://bis.gov.in/"),
    ("Web Source", "https://ncdrc.nic.in/"),
    ("Web Source", "https://www.franchise.org/franchise-information/franchise-trends"),
    ("Web Source", "https://labour.gov.in/occupational-safety-and-health"),
    ("Web Source", "https://www.fssai.gov.in/cms/regulations.php"),
    ("Web Source", "https://www.india.gov.in/topics/industries/food-processing"),
    ("Web Source", "https://www.osha.gov/young-workers"),
    ("Web Source", "https://labour.gov.in/minimum-wages-act"),
    ("Web Source", "https://www.coffee.org.in/"),
    ("Web Source", "https://www.franchise.org/franchise-information/buying-a-franchise"),
    ("Web Source", "https://www.fssai.gov.in/cms/rules.php"),
    ("Web Source", "https://www.indianspices.com/"),
    ("Web Source", "https://www.osha.gov/fall-protection"),
    ("Web Source", "https://www.franchise.org/franchise-opportunities/food"),
    ("Web Source", "https://labour.gov.in/factories-act"),
    ("Web Source", "https://labour.gov.in/payment-of-wages-act"),
    ("Web Source", "https://mpeda.gov.in/"),
    ("Web Source", "https://www.servsafe.com/food-handler"),
    ("Web Source", "https://www.cdc.gov/foodsafety/prevention.html"),
    ("Web Source", "https://www.franchise.org/franchise-information/starting-a-franchise"),
    ("Web Source", "https://www.osha.gov/ergonomics"),
    ("Web Source", "https://www.osha.gov/retail-facilities"),
    ("Web Source", "https://labour.gov.in/child-labour"),
    ("Web Source", "https://www.iso.org/iso-22000-food-safety-management.html"),
    ("Web Source", "https://www.sqfi.com/"),
    ("Web Source", "https://www.osha.gov/heat-exposure"),
    ("Web Source", "https://www.osha.gov/hazard-communication"),
    ("Food Safety",       "https://www.who.int/publications/i/item/9789241549691"),
    ("Food Safety",       "https://www.fao.org/food-safety/resources/en/"),
    ("Food Safety",       "https://fssai.gov.in/cms/food-safety-and-standards-regulations.php"),
    ("Food Safety",       "https://fssai.gov.in/cms/resources.php"),
    ("Food Safety",       "https://www.fda.gov/food/guidance-documents-regulatory-information-topic/guidance-documents-food-labeling"),
    ("Food Standards",    "https://www.fao.org/fao-who-codexalimentarius/codex-texts/codes-of-practice/en/"),
    ("Food Standards",    "https://www.fao.org/fao-who-codexalimentarius/codex-texts/list-standards/en/"),
    ("HR Management",     "https://www.ilo.org/global/publications/lang--en/index.htm"),
    ("HR Management",     "https://www.ilo.org/wcmsp5/groups/public/---ed_emp/---emp_ent/documents/"),
    ("India Compliance",  "https://labour.gov.in/publications"),
    ("India Compliance",  "https://fssai.gov.in/upload/uploadfiles/files/"),
    ("India Compliance",  "https://mofpi.gov.in/resources/publications"),
    ("Supply Chain",      "https://www.fao.org/food-loss-and-food-waste/resources/en/"),
    ("Marketing",         "https://unctad.org/topic/ecommerce-and-digital-economy/e-commerce"),
    ("Financial",         "https://www.cbic.gov.in/resources//htdocs-cbec/gst/"),
    ("Financial",         "https://rbidocs.rbi.org.in/rdocs/Publications/"),
    ("Quality",           "https://www.unido.org/sites/default/files/2009-05/"),
    ("Sustainability",    "https://www.fao.org/sustainable-food-systems/en/"),
    ("Research",          "https://arxiv.org/search/?searchtype=all&query=retail+inventory+machine+learning"),
    ("Research",          "https://arxiv.org/search/?searchtype=all&query=employee+attrition+prediction"),
    ("Research",          "https://arxiv.org/search/?searchtype=all&query=customer+satisfaction+NLP"),
    ("Research",          "https://arxiv.org/search/?searchtype=all&query=franchise+operations+analytics"),
    ("Research",          "https://arxiv.org/search/?searchtype=all&query=demand+forecasting+deep+learning+retail"),
    ("Research",          "https://arxiv.org/search/?searchtype=all&query=anomaly+detection+time+series"),
    ("Research",          "https://arxiv.org/search/?searchtype=all&query=RAG+retrieval+augmented+generation"),
    ("Research",          "https://arxiv.org/search/?searchtype=all&query=knowledge+graph+enterprise"),
    ("Research",          "https://arxiv.org/search/?searchtype=all&query=marketing+ROI+optimization"),
    ("World Bank",        "https://openknowledge.worldbank.org/search?query=franchise+india+retail"),
    ("World Bank",        "https://openknowledge.worldbank.org/search?query=food+safety+supply+chain"),
    ("World Bank",        "https://openknowledge.worldbank.org/search?query=msme+india+workforce"),
    ("OECD",              "https://www.oecd.org/industry/smes/"),
    ("OECD",              "https://www.oecd.org/consumer/"),
    ("Training",          "https://nsdcindia.org/resources/"),
    ("Training",          "https://www.msde.gov.in/en/publications"),
    ("IFC",               "https://www.ifc.org/en/insights-reports"),
    ("India Statistics",  "https://mospi.gov.in/publication"),
    ("NABARD",            "https://www.nabard.org/content.aspx?id=572"),
    ("RBI",               "https://rbidocs.rbi.org.in/rdocs/Publications/"),
    ("SIDBI",             "https://www.sidbi.in/en/home"),
    ("Codex",             "https://www.fao.org/fao-who-codexalimentarius/codex-texts/dbs/CXS/en/"),
    ("ILO",               "https://www.ilo.org/wcmsp5/groups/public/---ed_dialogue/---sector/documents/"),
    ("APEDA",             "https://apeda.gov.in/apedawebsite/six_head_product/"),


    # Additional official/public sources added for Milestone 3 Task 1 (India-first coverage).
    ("FSSAI Hygiene", "https://fssai.gov.in/inspection/hygiene-requirements"),
    ("FSSAI Guidance", "https://fssai.gov.in/business/guidance"),
    ("FSSAI Rules", "https://fssai.gov.in/food-law/rules-2011"),
    ("FSSAI Registration", "https://www.fssai.gov.in/business/registration"),
    ("FSSAI Regulations PDF", "https://www.fssai.gov.in/upload/uploadfiles/files/Final_Regulations_2010.pdf"),
    ("FSSAI Food Hygiene PDF", "https://www.fssai.gov.in/upload/uploadfiles/files/Guidance_Note_Food_Hygiene_Safety_07_06_2020_Revised_10_06_2020.pdf"),
    ("FoSTaC Startup Food Safety PDF", "https://fostac.fssai.gov.in/assets/docs/guide-for-food-startup.pdf"),
    ("India Labour Codes", "https://www.labour.gov.in/offerings/schemes-and-services/details/labour-codes-gzNzQzMtQWa"),
    ("CBIC GST Invoice Rules", "https://cbic-gst.gov.in/gst-invoice-rules.html"),
    ("CBIC Tax Invoice PDF", "https://cbic-gst.gov.in/pdf/e-version-gst-fliers/tax-invoice-efliers.pdf"),
    ("Consumer E-Commerce Rules PDF", "https://consumeraffairs.nic.in/sites/default/files/E%20commerce%20rules.pdf"),
    ("NPCI UPI Merchant FAQs", "https://www.npci.org.in/what-we-do/upi/faqs"),
    ("NPCI UPI Statistics", "https://www.npci.org.in/product/upi/product-statistics?financialYear=2026-27"),
    ("NPCI UPI Merchant Guidelines PDF", "https://www.npci.org.in/uploads/BHIM_UPI_Guidelines_2026_012a0b1bce.pdf"),
    ("eNAM Resources", "https://enam.gov.in/resources"),
    ("MSME Schemes", "https://www.msme.gov.in/?q=schemes"),
    ("MSME Annual Report PDF", "https://www.msme.gov.in/sites/default/files/MSME-ANNUAL-REPORT-2024-25-ENGLISH.pdf"),
    ("MSME Scheme Booklet PDF", "https://my.msme.gov.in/files/assets/pdf/v2.0%20-2025-%20Scheme%20booklet%20Eng-14.02.2025.pdf"),
    ("EPFO Employer Guide PDF", "https://www.epfindia.gov.in/site_docs/PDFs/MiscPDFs/Employer_Information_Booklet.pdf"),
    ("EPFO Employer Services", "https://www.epfindia.gov.in/site_en/For_Employers.php"),
    ("ESIC Employer Registration PDF", "https://portal.esic.gov.in/ESICInsurance1/ESICInsurancePortal/Documents/Employer_Employee_registration_through_portal.pdf"),
    ("GST New Taxpayer Welcome Kit PDF", "https://tutorial.gst.gov.in/downloads/news/welcome_kit_for_new_taxpyers.pdf"),
    ("GST e-Invoice User Guide PDF", "https://tutorial.gst.gov.in/einvoice/files/FAQs_Manual_e-Invoice%20JSON%20download%20functionality.pdf"),
    ("NIC GST e-Invoice", "https://www.nic.gov.in/project/gst-e-invoice/"),
    ("APEDA Quality Development", "https://apeda.gov.in/quality-development"),
    ("APEDA Quality Development Guidelines PDF", "https://www.apeda.gov.in/sites/default/files/2025-02/FAS_Guidelines_05102021.pdf"),
    ("NHB Cold Chain", "https://nhb.gov.in/"),
    ("NHB Cold Chain News and Guidelines", "https://nhb.gov.in/NewsAnnouncement.aspx"),
    ("NHB Cold Chain Engineering Guidelines PDF", "https://www.nhb.gov.in/guideline/ENGINEERINGGUIDELINES2025.pdf"),
    ("NHB Post-Harvest Storage Manual PDF", "https://nhb.gov.in/documents/storage-manual.pdf"),
    ("India Code Food Safety Act", "https://www.indiacode.nic.in/indiacode/handle/123456789/2027?col=123456789%2F1362&view_type=search"),
    ("India Code Consumer Protection Act PDF", "https://www.indiacode.nic.in/indiacode/bitstream/123456789/15256/1/eng201935.pdf"),
    ("GST Council Central GST Laws", "https://gstcouncil.gov.in/central-gst"),
    ("Income Tax Department Tax Audit Guidance", "https://www.incometax.gov.in/iec/foportal/help/all-topics/e-filing-services/income-tax-forms"),
    ("MCA Companies Act PDF", "https://www.mca.gov.in/Ministry/pdf/CompaniesAct2013.pdf"),
    ("MCA E-Book FAQ PDF", "https://www.mca.gov.in/Ministry/pdf/EBookFAQ.pdf"),
    ("CII Retail and Consumer Sector Update", "https://www.cii.in/ResourceDetails.aspx?enc=prvePUj2bdMtgTmvPwvisYH+5EnGjyGXO9hLECvTuNukOMZZOgXxJ4SAnAYiiGyh+2UDFx5GcYGh2+dfLekPHA%3D%3D"),
    ("Agmarknet Market Data SOP PDF", "https://www.agmarknet.gov.in/DailyDataEntry/Docs/SOP_DEO.pdf"),
]




# ─────────────────────────────────────────────────────────────────────────────
# HELPERS
# ─────────────────────────────────────────────────────────────────────────────

def is_pdf_url(url):
    low = url.lower()
    return (low.endswith(".pdf")
            or "/pdf/" in low
            or "download=pdf" in low
            or ".pdf?" in low
            or "format=pdf" in low)


def same_domain(url, base):
    try:
        a = urllib.parse.urlparse(url).netloc.lower().removeprefix("www.")
        b = urllib.parse.urlparse(base).netloc.lower().removeprefix("www.")
        return a == b
    except Exception:
        return False


def title_from_url(url):
    name = url.rstrip("/").split("/")[-1]
    # strip extension safely without regex escape warning
    for ext in [".pdf", ".PDF", ".htm", ".html", ".aspx", ".php"]:
        if name.endswith(ext):
            name = name[: -len(ext)]
    return name.replace("-", " ").replace("_", " ").strip()[:100]


def discover_pdfs(url, category, visited=None, found=None,
                  max_depth=MAX_CRAWL_DEPTH, max_per_seed=MAX_PDFS_PER_SEED):
    if visited is None:
        visited = set()
    if found is None:
        found = []

    # ArXiv special handling
    if "arxiv.org/search" in url:
        return _crawl_arxiv(url, category, max_per_seed)

    queue = deque([(url, 0)])
    visited.add(url)

    while queue and len(found) < max_per_seed:
        cur_url, depth = queue.popleft()
        try:
            r = requests.get(cur_url, headers=HEADERS,
                             timeout=REQUEST_TIMEOUT, allow_redirects=True)
            if r.status_code != 200:
                continue
            ctype = r.headers.get("content-type", "")

            if "pdf" in ctype.lower():
                if cur_url not in [f["url"] for f in found]:
                    found.append({"url": cur_url, "category": category,
                                  "title": title_from_url(cur_url)})
                continue

            soup = BeautifulSoup(r.text, "lxml")

            for tag in soup.find_all("a", href=True):
                href = tag["href"].strip()
                abs_href = urllib.parse.urljoin(cur_url, href)
                if is_pdf_url(abs_href) and abs_href not in [f["url"] for f in found]:
                    title = (tag.get_text(strip=True) or title_from_url(abs_href))[:120]
                    found.append({"url": abs_href, "category": category, "title": title})
                    if len(found) >= max_per_seed:
                        break

            if depth < max_depth:
                for tag in soup.find_all("a", href=True):
                    href = tag["href"].strip()
                    abs_href = urllib.parse.urljoin(cur_url, href)
                    if (abs_href not in visited
                            and same_domain(abs_href, url)
                            and not is_pdf_url(abs_href)
                            and abs_href.startswith("http")
                            and "#" not in abs_href
                            and len(visited) < MAX_CRAWL_PAGES):
                        visited.add(abs_href)
                        queue.append((abs_href, depth + 1))

        except Exception:
            pass
        time.sleep(DELAY_BETWEEN)

    return found


def _crawl_arxiv(search_url, category, limit=30):
    found = []
    try:
        r = requests.get(search_url, headers=HEADERS, timeout=REQUEST_TIMEOUT)
        soup = BeautifulSoup(r.text, "lxml")
        for li in soup.select("li.arxiv-result")[:limit]:
            title_tag = li.select_one("p.title")
            pdf_link  = li.select_one('a[href*="/pdf/"]')
            if pdf_link:
                pdf_url = pdf_link["href"]
                if "arxiv.org" not in pdf_url:
                    pdf_url = "https://arxiv.org" + pdf_url
                if not pdf_url.endswith(".pdf"):
                    pdf_url += ".pdf"
                title = title_tag.get_text(strip=True)[:120] if title_tag else title_from_url(pdf_url)
                found.append({"url": pdf_url, "category": category, "title": title})
    except Exception:
        pass
    return found


def download_pdf(entry, out_dir):
    url = entry["url"]
    title = entry["title"]
    url_hash = hashlib.sha256(url.encode("utf-8")).hexdigest()[:12]
    safe = "".join(c if c.isalnum() or c in " _-" else "_"
                    for c in f"{entry['category']}___{title}")[:80].strip(" _")
    fname = f"{safe or 'document'}_{url_hash}.pdf"
    fpath = out_dir / fname

    if fpath.exists() and fpath.stat().st_size > 1024:
        try:
            if fpath.open("rb").read(5) == b"%PDF-":
                return True, fpath, "cached"
        except Exception:
            pass
        fpath.unlink(missing_ok=True)

    try:
        with requests.get(url, headers=PDF_HEADERS, timeout=DOWNLOAD_TIMEOUT,
                          allow_redirects=True, stream=True) as r:
            if r.status_code != 200:
                return False, fpath, "http_" + str(r.status_code)
            chunks, total = [], 0
            for part in r.iter_content(1024 * 64):
                if not part:
                    continue
                total += len(part)
                if total > MAX_PDF_BYTES:
                    return False, fpath, "too_large"
                chunks.append(part)
            content = b"".join(chunks)
            if not content.startswith(b"%PDF-"):
                return False, fpath, "not_pdf_content"
            fpath.write_bytes(content)
            entry["sha256"] = hashlib.sha256(content).hexdigest()
            entry["bytes"] = len(content)
            return True, fpath, "downloaded"
    except Exception as e:
        return False, fpath, "err_" + str(e)[:60]


# ─────────────────────────────────────────────────────────────────────────────
# TEXT EXTRACTION + CHUNKING
# ─────────────────────────────────────────────────────────────────────────────

def extract_text(fpath):
    try:
        import fitz
        doc  = fitz.open(str(fpath))
        text = "\n".join(p.get_text() for p in doc)
        doc.close()
        return text
    except Exception:
        pass
    try:
        raw  = fpath.read_bytes()
        soup = BeautifulSoup(raw, "lxml")
        return soup.get_text(separator="\n", strip=True)
    except Exception:
        return ""


def chunk_text(text, title, source_url,
               chunk_words=CHUNK_WORDS, overlap=CHUNK_OVERLAP):
    words  = text.split()
    step   = chunk_words - overlap
    chunks = []
    for i in range(0, max(1, len(words) - overlap), step):
        piece = " ".join(words[i: i + chunk_words])
        if len(piece) > 80:
            cid = hashlib.md5((source_url + str(i)).encode()).hexdigest()[:12]
            chunks.append({
                "chunk_id": cid,
                "text":     piece,
                "title":    title,
                "source":   source_url,
                "offset":   i,
            })
    return chunks


# ─────────────────────────────────────────────────────────────────────────────
# RAG INDEX BUILD  (matches existing RAG notebook format exactly)
# ─────────────────────────────────────────────────────────────────────────────

def build_rag_index(chunks, out_base, project_key):
    import numpy as np
    import faiss
    from sentence_transformers import SentenceTransformer
    from rank_bm25 import BM25Okapi

    faiss_dir = out_base / "faiss_indexes"
    bm25_dir  = out_base / "bm25_indexes"
    faiss_dir.mkdir(parents=True, exist_ok=True)
    bm25_dir.mkdir(parents=True, exist_ok=True)

    ST_CACHE = "/content/.cache/sentence_transformers"
    print("  Loading sentence-transformer (all-MiniLM-L6-v2)...")
    model = SentenceTransformer("all-MiniLM-L6-v2", cache_folder=ST_CACHE)

    texts = [c["text"] for c in chunks]
    print(f"  Encoding {len(texts)} chunks...")
    embs  = model.encode(texts, show_progress_bar=True,
                         batch_size=64, convert_to_numpy=True).astype("float32")

    faiss.normalize_L2(embs)
    index = faiss.IndexFlatIP(embs.shape[1])
    index.add(embs)

    idx_path  = str(faiss_dir / (project_key + "_faiss.index"))
    meta_path = str(faiss_dir / (project_key + "_chunks_meta.json"))
    bm25_path = str(bm25_dir  / (project_key + "_bm25.pkl"))

    faiss.write_index(index, idx_path)
    with open(meta_path, "w") as f:
        json.dump(chunks, f)

    tokenized = [c["text"].lower().split() for c in chunks]
    bm25      = BM25Okapi(tokenized)
    with open(bm25_path, "wb") as f:
        pickle.dump({"bm25": bm25, "chunks": chunks}, f)

    print("  FAISS index:  " + idx_path)
    print("  Chunk meta:   " + meta_path)
    print("  BM25 index:   " + bm25_path)
    return index.ntotal


# ─────────────────────────────────────────────────────────────────────────────
# MAIN RUNNER
# ─────────────────────────────────────────────────────────────────────────────

def run_project(project_name, project_key, seeds, out_base_str):
    out_base = Path(out_base_str)
    pdf_dir  = out_base / "rag_pdfs"
    pdf_dir.mkdir(parents=True, exist_ok=True)

    print("\n" + "=" * 65)
    print("  PROJECT: " + project_name)
    print("  Seed sites: " + str(len(seeds)))
    print("  Output: " + str(out_base))
    print("=" * 65)

    # Step 1 — crawl
    print("\n  [1/4] Crawling " + str(len(seeds)) + " seed URLs for PDF documents...")
    all_found = []
    visited_gl = set()
    for cat, seed_url in tqdm(seeds, desc="  Crawling"):
        found = discover_pdfs(seed_url, cat, visited=visited_gl)
        all_found.extend(found)
    print("  Unique HTML pages visited: " + str(len(visited_gl)))

    # Deduplicate
    seen, unique = set(), []
    for e in all_found:
        if e["url"] not in seen:
            seen.add(e["url"])
            unique.append(e)
    print("\n  Unique PDFs discovered: " + str(len(unique)))

    # Step 2 — download
    print("\n  [2/4] Downloading " + str(len(unique)) + " PDFs...")
    manifest = {
        "project": project_name,
        "generated_at": datetime.now().isoformat(),
        "total_discovered": len(unique),
        "downloaded": 0, "cached": 0, "failed": 0,
        "target_pdf_count": TARGET_PDF_COUNT,
        "seed_url_count": len(seeds),
        "unique_domains": len({urllib.parse.urlparse(u).netloc.lower().removeprefix("www.") for _, u in seeds}),
        "files": [],
    }
    for entry in tqdm(unique, desc="  Downloading"):
        ok, fpath, status = download_pdf(entry, pdf_dir)
        entry["local_path"] = str(fpath)
        entry["status"]     = status
        if ok:
            if status == "cached":
                manifest["cached"] += 1
            else:
                manifest["downloaded"] += 1
            try:
                entry["sha256"] = hashlib.sha256(fpath.read_bytes()).hexdigest()
                entry["bytes"] = fpath.stat().st_size
            except Exception:
                pass
        else:
            manifest["failed"] += 1
        manifest["files"].append(entry)
        time.sleep(DELAY_BETWEEN)

    manifest_path = out_base / "manifest.json"
    with open(manifest_path, "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2, ensure_ascii=False)

    # Human-readable source inventory for milestone evidence and auditing.
    try:
        import csv
        with open(out_base / "source_inventory.csv", "w", newline="", encoding="utf-8") as f:
            fields = ["category", "title", "url", "status", "local_path", "bytes", "sha256"]
            w = csv.DictWriter(f, fieldnames=fields)
            w.writeheader()
            for e in manifest["files"]:
                w.writerow({k: e.get(k, "") for k in fields})
    except Exception as e:
        print("  Could not write source inventory:", e)

    successful = manifest["downloaded"] + manifest["cached"]
    # Count valid PDFs already present in the destination as well as this run's downloads.
    valid_folder_files = []
    folder_hashes = set()
    for candidate in pdf_dir.glob("*.pdf"):
        try:
            if candidate.stat().st_size > 1024 and candidate.open("rb").read(5) == b"%PDF-":
                valid_folder_files.append(candidate)
                folder_hashes.add(hashlib.sha256(candidate.read_bytes()).hexdigest())
        except Exception:
            continue
    manifest["valid_pdf_files_in_folder"] = len(valid_folder_files)
    manifest["unique_pdf_content_in_folder"] = len(folder_hashes)
    print("  Downloaded: " + str(manifest["downloaded"]) +
          " | Cached from current discovery: " + str(manifest["cached"]) +
          " | Failed: " + str(manifest["failed"]))
    print("  Valid PDFs in rag_pdfs folder: " + str(len(valid_folder_files)))
    print("  Unique PDF contents in folder: " + str(len(folder_hashes)))
    if len(folder_hashes) < TARGET_PDF_COUNT:
        print("  WARNING: TARGET NOT MET. Need " + str(TARGET_PDF_COUNT - len(folder_hashes)) +
              " more unique valid PDFs. Add official source seeds or increase MAX_CRAWL_PAGES / MAX_PDFS_PER_SEED, then rerun.")
    else:
        print("  TARGET MET: at least " + str(TARGET_PDF_COUNT) + " unique valid PDF contents.")
    with open(manifest_path, "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2, ensure_ascii=False)

    # Step 3 — chunk
    print("\n  [3/4] Extracting text and chunking...")
    all_chunks = []
    ok_files = [e for e in manifest["files"]
                if e.get("status") in ("downloaded", "cached")]
    seen_hashes = set()
    for entry in tqdm(ok_files, desc="  Chunking"):
        fpath = Path(entry.get("local_path", ""))
        if not fpath.exists() or fpath.stat().st_size < 512:
            continue
        try:
            if fpath.open("rb").read(5) != b"%PDF-":
                continue
        except Exception:
            continue
        digest = entry.get("sha256")
        if digest and digest in seen_hashes:
            entry["duplicate_content"] = True
            continue
        if digest:
            seen_hashes.add(digest)
        text = extract_text(fpath)
        if len(text.split()) < 20:
            entry["extract_status"] = "too_little_text"
            continue
        entry["extract_status"] = "extracted"
        entry["word_count"] = len(text.split())
        all_chunks.extend(chunk_text(text, entry["title"], entry["url"]))
    print("  Unique PDF content files extracted: " + str(len(seen_hashes)))
    print("  Total chunks: " + str(len(all_chunks)))
    # Persist extraction/chunk statistics into the manifest.
    manifest["unique_content_pdfs"] = len(seen_hashes)
    manifest["chunk_count"] = len(all_chunks)
    with open(manifest_path, "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2, ensure_ascii=False)

    # Step 4 — index
    print("\n  [4/4] Building FAISS + BM25 indexes...")
    n = 0
    if all_chunks:
        n = build_rag_index(all_chunks, out_base, project_key)
    else:
        print("  No chunks to index")
    print("\n  " + project_name + " COMPLETE: " + str(n) + " vectors")
    print("  Evidence files: " + str(out_base / "manifest.json") +
          " and " + str(out_base / "source_inventory.csv"))
    return manifest


# ─────────────────────────────────────────────────────────────────────────────
# ENTRY POINT
# ─────────────────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    DRIVE = Path("/content/drive/MyDrive")

    if DRIVE.exists():
        FC_BASE = str(DRIVE / FC_DRIVE_FOLDER)

        print("Google Drive mounted.")
        print("FranchiseOps -> " + FC_BASE)

    else:
        FC_BASE = "/content/FranchiseOps_AI"

        print("Drive not mounted — saving locally.")

    print("FranchiseOps seed sites: " + str(len(FRANCHISE_SEEDS)))


    fc_m = run_project("FranchiseOps AI", "franchise", FRANCHISE_SEEDS, FC_BASE)


    print("\n" + "=" * 65)
    print("  FINAL SUMMARY")
    print("=" * 65)
    for nm, m in [("FranchiseOps", fc_m)]:
        print("  " + nm.ljust(14) +
              " | Found: " + str(m["total_discovered"]) +
              " | Valid PDFs: " + str(m["downloaded"] + m["cached"]) +
              " | Unique content PDFs: " + str(m.get("unique_content_pdfs", 0)) +
              " | Chunks: " + str(m.get("chunk_count", 0)) +
              " | Failed: " + str(m["failed"]))
    print("=" * 65)


Writing /content/pdf_web_crawler.py


In [3]:
!pip install -q requests beautifulsoup4 lxml tqdm PyMuPDF sentence-transformers faiss-cpu rank-bm25

from google.colab import drive
import os
if not os.path.isdir('/content/drive/MyDrive'):
    drive.mount('/content/drive')
else:
    print('Google Drive already mounted.')

# Verify the intended output path exists before starting the crawler.
os.makedirs('/content/drive/MyDrive/FranchiseOps_AI', exist_ok=True)
!python3 /content/pdf_web_crawler.py


Google Drive already mounted.
Google Drive mounted.
FranchiseOps -> /content/drive/MyDrive/FranchiseOps_AI
FranchiseOps seed sites: 145

  PROJECT: FranchiseOps AI
  Seed sites: 145
  Output: /content/drive/MyDrive/FranchiseOps_AI

  [1/4] Crawling 145 seed URLs for PDF documents...
  Crawling: 100% 145/145 [08:00<00:00,  3.31s/it]
  Unique HTML pages visited: 1321

  Unique PDFs discovered: 574

  [2/4] Downloading 574 PDFs...
  Downloading: 100% 574/574 [19:00<00:00,  1.99s/it]
  Downloaded: 380 | Cached from current discovery: 163 | Failed: 31
  Valid PDFs in rag_pdfs folder: 543
  Unique PDF contents in folder: 528
  TARGET MET: at least 150 unique valid PDF contents.

  [3/4] Extracting text and chunking...
  Chunking:   0% 0/543 [00:00<?, ?it/s]/usr/local/lib/python3.13/dist-packages/fitz/__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(
  Chunking: 100% 543/543 [01:21<00:00,  6.64it/s]
  Uniq

# FranchiseOps RAG Builder — Milestone 3 Task 1

**Goal:** collect at least 150 valid public PDFs, extract and deduplicate content, chunk the text, build FAISS + BM25 indexes, and write the source-attributed corpus to Google Drive for the final app.

**Run order matters:** this notebook mounts Google Drive in the crawler cell *before* launching the crawler. Do not move the crawler execution above the Drive mount.

**Target evidence:** `manifest.json`, `source_inventory.csv`, `pdf_document_catalog.csv`, `synthetic_kb/all_chunks.jsonl`, FAISS/BM25 index files, and the printed PDF/chunk counts. Do not report 150+ until the validation output confirms at least 150 unique valid PDF files.

The seed list combines the mentor's sources with additional public sources, prioritising Indian food safety, labour, GST, MSME, EPFO/ESIC, payments, consumer protection and supply-chain material. Respect source terms, robots policies and rate limits; only ingest public documents and do not collect personal data.

The added seed block contains 38 new URLs from 21 additional host domains (counting host names after removing www); the crawler still verifies which of them actually return usable PDFs. Some links are direct PDF seeds, while others are web pages that expose PDFs.


In [4]:
!pip install -q faiss-cpu sentence-transformers rank-bm25 pymupdf
!pip install -q langchain langchain-community langchain-text-splitters
!pip install -q plotly pandas numpy scikit-learn joblib faker vaderSentiment
print('All dependencies installed')


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
All dependencies installed


In [5]:
from google.colab import drive
import os
if not os.path.isdir('/content/drive/MyDrive'):
    drive.mount('/content/drive')
else:
    print('Google Drive already mounted.')

DRIVE_BASE = '/content/drive/MyDrive/FranchiseOps_AI'
FAISS_DIR  = f'{DRIVE_BASE}/faiss_indexes'
BM25_DIR   = f'{DRIVE_BASE}/bm25_indexes'
ML_DIR     = f'{DRIVE_BASE}/ml_models'
PDF_DIR    = f'{DRIVE_BASE}/rag_pdfs'
KB_DIR     = f'{DRIVE_BASE}/synthetic_kb'
DB_PATH    = f'{DRIVE_BASE}/franchise_database.db'
LLM_CACHE  = '/content/.cache/hf_models'
ST_CACHE   = '/content/.cache/sentence_transformers'

for d in [FAISS_DIR, BM25_DIR, ML_DIR, PDF_DIR, KB_DIR, LLM_CACHE, ST_CACHE]:
    os.makedirs(d, exist_ok=True)

print('Drive mounted. Storage layout:')
print(f'  FAISS index   -> {FAISS_DIR}')
print(f'  BM25 index    -> {BM25_DIR}')
print(f'  ML models     -> {ML_DIR}')
print(f'  PDF uploads   -> {PDF_DIR}')
print(f'  Knowledge KB  -> {KB_DIR}')
print(f'  Database      -> {DB_PATH}')
print(f'  LLM cache     -> {LLM_CACHE}  (local only - too large for Drive)')


Google Drive already mounted.
Drive mounted. Storage layout:
  FAISS index   -> /content/drive/MyDrive/FranchiseOps_AI/faiss_indexes
  BM25 index    -> /content/drive/MyDrive/FranchiseOps_AI/bm25_indexes
  ML models     -> /content/drive/MyDrive/FranchiseOps_AI/ml_models
  PDF uploads   -> /content/drive/MyDrive/FranchiseOps_AI/rag_pdfs
  Knowledge KB  -> /content/drive/MyDrive/FranchiseOps_AI/synthetic_kb
  Database      -> /content/drive/MyDrive/FranchiseOps_AI/franchise_database.db
  LLM cache     -> /content/.cache/hf_models  (local only - too large for Drive)


In [6]:
import json

DOCS = [
    {'title': 'SOP-001: Customer Service Standards',
     'content': 'All franchise outlets must maintain minimum CSAT score of 4.0/5.0. Staff must greet every customer within 30 seconds. Complaint resolution must be completed within 24 hours. Monthly mystery shopping audits are conducted at all Tier 1 outlets. Customer feedback cards are collected at every transaction and digitised weekly. Any CSAT below 3.5 triggers immediate branch manager review. Digital feedback via QR codes increases response rate by 4x versus paper forms.'},
    {'title': 'SOP-002: Inventory Management Protocol',
     'content': 'Inventory reorder must trigger automatically when stock falls below 20% of monthly demand. Weekly stock audits are mandatory for Food and Beverage categories. Expired stock must be quarantined and disposed within 48 hours. FIFO (First In First Out) must be followed for all perishable items. Supplier lead time must be accounted in reorder calculations. AI-driven demand forecasting reduces wastage by 23% and stockout incidents by 41%. Optimal reorder point = Average Daily Demand x Lead Time + Safety Stock.'},
    {'title': 'SOP-003: Staff Attrition Management',
     'content': 'Staff attrition above 15% per quarter requires immediate HR intervention and root cause analysis. Exit interviews are mandatory for all departing employees. Job satisfaction surveys are administered quarterly. High performers with job_satisfaction >= 4 and tenure > 2 years are eligible for Fast Track Promotion. Overtime hours must not exceed 20 hours per month per employee. Work-life balance scores correlate 0.74 with 12-month retention. Monthly 1:1 meetings between manager and staff improve satisfaction by 18%.'},
    {'title': 'SOP-004: Marketing Campaign Compliance',
     'content': 'All marketing campaigns must be approved by Regional Marketing Manager before launch. Campaigns must achieve minimum ROI of 1.5x to be considered successful. Digital channels must follow brand guidelines including logo usage, color palettes, and tone. Social media posts require 24-hour approval window. Campaign performance is reviewed weekly. Budget allocation recommended: 40% Digital, 25% Social Media, 20% Influencer, 15% Print. Digital channels deliver highest ROI at 3.2x average.'},
    {'title': 'SOP-005: Audit and Compliance Framework',
     'content': 'Audit compliance score below 70 triggers mandatory corrective action plan within 72 hours. Hygiene audits are conducted monthly by internal QA team. Safety compliance checks occur bi-weekly. Financial audits are quarterly. Brand standards audit happens annually. Failed audits require re-audit within 30 days. Consecutive failures result in franchise review board escalation. Pre-audit self-assessment using 45-point checklist catches 78% of issues before formal review.'},
    {'title': 'SOP-006: Outlet Tier Classification',
     'content': 'Tier 1 outlets: Revenue above 150000 INR per month, CSAT >= 4.3, Staff >= 12 employees. Tier 2 outlets: Revenue 80000-150000 INR per month, CSAT >= 3.8, Staff 6-12. Tier 3 outlets: Revenue below 80000 INR per month. Classification reviewed quarterly. Tier upgrades require 3 consecutive quarters of performance above threshold. Tier 1 outlets receive priority marketing support and dedicated account manager.'},
    {'title': 'SOP-007: Digital Transformation Requirements',
     'content': 'All outlets must use the FranchiseOps AI platform for inventory tracking, staff scheduling, and sales reporting. Paper-based systems must be fully digitized by Q4. POS system integration is mandatory for all Tier 1 and Tier 2 outlets. Real-time sales data must sync to central database every 15 minutes. Outlets failing digital compliance face penalty points in annual review.'},
    {'title': 'SOP-008: Health and Safety Standards',
     'content': 'Temperature logs for refrigerated items must be recorded every 4 hours. Commercial refrigerators must maintain temperature at or below 4 degrees Celsius. All food handlers must possess valid food safety certification renewed annually. Emergency evacuation procedures must be posted and drilled quarterly. First aid kits must be fully stocked and inspected monthly. CCTV systems must have 30-day footage retention minimum.'},
    {'title': 'SOP-009: Financial Management and Reporting',
     'content': 'Daily revenue must be reconciled and submitted to Regional Finance by 11 PM. Cash variance greater than 2% triggers immediate investigation. P&L reports are submitted monthly on the 5th business day. Operating cost ratio must not exceed 65% of revenue. Industry benchmark operating cost ratio is 58-62%. Labor cost should not exceed 28% of revenue. Capital expenditure above 50000 INR requires Franchise Owner approval.'},
    {'title': 'SOP-010: Crisis Management Protocol',
     'content': 'Food safety incidents must be reported to Regional Manager within 1 hour. Customer injury claims must be escalated to legal team immediately. Media inquiries must be routed to Corporate Communications. Outlet closure decisions rest with Regional Director. Post-crisis report must be submitted within 7 days. All incidents are logged in the FranchiseOps AI alert system with resolution tracking.'},
    {'title': 'BP-001: Revenue Growth Strategies',
     'content': 'Top-performing franchise outlets achieve revenue growth through three pillars: upselling training increases average transaction value by 18%, loyalty program enrollment means customers spend 2.3x more, digital ordering channels reduce wait time by 40% and increase throughput by 25%. Weekend promotions consistently yield 30% higher revenue versus weekday baselines. Festival season October to January typically 40% above monthly average.'},
    {'title': 'BP-002: Staff Retention Playbook',
     'content': 'Outlets with lowest attrition share these practices: structured onboarding in first 30 days, monthly 1-on-1 meetings between manager and each staff member, peer recognition programs, clear promotion pathways at 6 months and 18 months, and competitive compensation reviewed annually. Outlets with AI-powered scheduling save 12% on labor costs versus manual scheduling.'},
    {'title': 'BP-003: Inventory Optimization Techniques',
     'content': 'AI-powered demand forecasting reduces wastage by 23% and stockout incidents by 41%. Vendor consolidation to 3-5 primary suppliers reduces procurement costs by 12%. Safety stock should be 15% of weekly demand for fast movers and 30% for seasonal items. FIFO is critical for perishables. Real-time inventory visibility prevents phantom stockouts and over-ordering.'},
    {'title': 'BP-004: Customer Experience Excellence',
     'content': 'CSAT above 4.5 is associated with 67% higher likelihood of repeat visits. Key drivers of positive CSAT: speed of service accounts for 35% of score, product quality 30%, staff attitude 25%, cleanliness 10%. Mystery shopping scores below 75 require immediate manager coaching. NPS target is 45 plus versus industry average of 32.'},
    {'title': 'BP-005: Marketing ROI Maximization',
     'content': 'Digital channels deliver highest ROI at 3.2x average. Email marketing to loyalty database achieves 28% open rate versus 18% industry average. Geo-targeted promotions within 3km radius convert at 2.1x versus broad campaigns. Budget allocation: 40% Digital, 25% Social, 20% Influencer, 15% Print. Influencer marketing with micro-influencers under 50000 followers delivers 7x higher engagement than celebrity endorsements.'},
    {'title': 'KPI-001: Revenue Benchmarks',
     'content': 'National average revenue per outlet is 95000 INR per month. Top quartile threshold is 140000 INR per month. Bottom quartile threshold is 55000 INR per month. Mumbai Delhi Bangalore outlets average 35% higher revenue than Tier 2 cities. Peak revenue months October to January during festival season typically 40% above monthly average. Lowest revenue June to August monsoon season.'},
    {'title': 'KPI-002: Operational Efficiency Benchmarks',
     'content': 'Industry benchmark operating cost ratio is 58-62% of revenue. Best-in-class outlets achieve 48-52%. Labor cost should not exceed 28% of revenue. Food cost benchmark is 25-30% of F&B revenue. Energy costs 5-8% of revenue. Franchise fee typically 6-8% of gross revenue. Outlets with AI-powered scheduling save 12% on labor costs versus manual scheduling.'},
    {'title': 'KPI-003: Customer Metrics Benchmarks',
     'content': 'Industry average CSAT is 3.6 out of 5.0. FranchiseOps network average is 4.1. Target CSAT for Tier 1 is 4.3 or above and for Tier 2 is 4.0 or above. Net Promoter Score target is 45 versus industry average of 32. Customer return rate within 30 days target is 65% or above. Average transaction value target is 320 INR versus industry average of 260 INR.'},
    {'title': 'ML-001: Attrition Prediction Model',
     'content': 'The current application implements Logistic Regression for employee attrition risk. Features used by the code are salary, overtime hours, job satisfaction, tenure, work-life balance, age, and attendance percentage. It uses a fed HR training table only when at least 300 labelled records are available; otherwise it bootstraps labels from existing heuristic attrition probabilities. Metrics from the bootstrap path are not independent validation and must not be presented as real-world accuracy. The model is refreshed during application operation.'},
    {'title': 'ML-002: Revenue Prediction and Demand Forecasting',
     'content': 'The current automated inventory agent uses a weekday-seasonal linear trend to estimate near-term demand and days of stock cover. Prophet is an optional method on the per-outlet Inventory forecast page when the library is installed and selected; it is not the default method for the fleet-wide agent. The current application does not implement the Gradient Boosting revenue model or a validated R-squared of 0.79. Reorder actions are proposed through the policy and approval workflow; they are not universally auto-executed.'},
]

doc_path = f'{KB_DIR}/franchise_kb.jsonl'
with open(doc_path, 'w', encoding='utf-8') as f:
    for doc in DOCS:
        # These are demo-only assumptions, not public law or validated industry benchmarks.
        doc['source'] = 'FranchiseOps internal demo SOP pack (synthetic / illustrative)'
        doc['category'] = 'Internal demo policy'
        f.write(json.dumps(doc, ensure_ascii=False) + '\n')

print(f'Generated {len(DOCS)} franchise knowledge documents -> Drive')
for d in DOCS:
    print(f'  * {d["title"]}')


Generated 20 franchise knowledge documents -> Drive
  * SOP-001: Customer Service Standards
  * SOP-002: Inventory Management Protocol
  * SOP-003: Staff Attrition Management
  * SOP-004: Marketing Campaign Compliance
  * SOP-005: Audit and Compliance Framework
  * SOP-006: Outlet Tier Classification
  * SOP-007: Digital Transformation Requirements
  * SOP-008: Health and Safety Standards
  * SOP-009: Financial Management and Reporting
  * SOP-010: Crisis Management Protocol
  * BP-001: Revenue Growth Strategies
  * BP-002: Staff Retention Playbook
  * BP-003: Inventory Optimization Techniques
  * BP-004: Customer Experience Excellence
  * BP-005: Marketing ROI Maximization
  * KPI-001: Revenue Benchmarks
  * KPI-002: Operational Efficiency Benchmarks
  * KPI-003: Customer Metrics Benchmarks
  * ML-001: Attrition Prediction Model
  * ML-002: Revenue Prediction and Demand Forecasting


In [7]:
import os, json, hashlib, csv
from pathlib import Path
import fitz

pdf_docs = []
pdf_catalog = []
seen_pdf_hashes = set()

# Preserve the crawler's original URL/category metadata for citations in the final app.
manifest_path = os.path.join(DRIVE_BASE, 'manifest.json')
source_by_name = {}
if os.path.exists(manifest_path):
    with open(manifest_path, encoding='utf-8') as f:
        crawl_manifest = json.load(f)
    for item in crawl_manifest.get('files', []):
        lp = item.get('local_path')
        if lp:
            source_by_name[Path(lp).name] = item
else:
    crawl_manifest = {}

pdf_files = sorted(Path(PDF_DIR).glob('*.pdf'))
print(f'PDF files present in Drive: {len(pdf_files)}')

for p in pdf_files:
    try:
        raw = p.read_bytes()
        if not raw.startswith(b'%PDF-'):
            print(f'  Skipped non-PDF file: {p.name}')
            continue
        digest = hashlib.sha256(raw).hexdigest()
        meta = source_by_name.get(p.name, {})
        if digest in seen_pdf_hashes:
            print(f'  Duplicate content skipped: {p.name}')
            continue
        seen_pdf_hashes.add(digest)
        page_count = 0
        word_count = 0
        with fitz.open(p) as doc:
            page_count = len(doc)
            for page_num, page in enumerate(doc):
                text = page.get_text('text').strip()
                if len(text) > 100:
                    words = len(text.split())
                    word_count += words
                    pdf_docs.append({
                        'title': f"{meta.get('title') or p.stem} - page {page_num + 1}",
                        'content': text,
                        'source': meta.get('url') or f'Drive PDF: {p.name}',
                        'category': meta.get('category', 'Uploaded PDF'),
                        'file_name': p.name,
                        'sha256': digest,
                        'page': page_num + 1
                    })
        pdf_catalog.append({
            'file_name': p.name,
            'title': meta.get('title') or p.stem,
            'category': meta.get('category', 'Uploaded PDF'),
            'source_url': meta.get('url', ''),
            'sha256': digest,
            'pages': page_count,
            'extracted_words': word_count,
            'status': 'extracted' if word_count else 'no_extractable_text'
        })
        print(f'  Ingested: {p.name} | {page_count} pages | {word_count} words')
    except Exception as e:
        print(f'  Error: {p.name}: {e}')

with open(f'{KB_DIR}/pdf_extracted.jsonl', 'w', encoding='utf-8') as f:
    for d in pdf_docs:
        f.write(json.dumps(d, ensure_ascii=False) + '\n')

with open(f'{DRIVE_BASE}/pdf_document_catalog.csv', 'w', newline='', encoding='utf-8') as f:
    fields = ['file_name', 'title', 'category', 'source_url', 'sha256', 'pages', 'extracted_words', 'status']
    w = csv.DictWriter(f, fieldnames=fields)
    w.writeheader()
    w.writerows(pdf_catalog)

print('\nPDF extraction summary:')
print(f'  Unique valid PDF files: {len(pdf_catalog)}')
print(f'  Extracted page records: {len(pdf_docs)}')
print(f'  Total extracted words: {sum(x["extracted_words"] for x in pdf_catalog):,}')
if len(pdf_catalog) < 150:
    print(f'  TARGET NOT MET: {150 - len(pdf_catalog)} more unique PDFs are needed.')
else:
    print('  TARGET MET: 150+ unique PDF files extracted.')
print(f'  Source catalogue: {DRIVE_BASE}/pdf_document_catalog.csv')


PDF files present in Drive: 543


/usr/local/lib/python3.13/dist-packages/fitz/__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(


  Ingested: APEDA Quality Development Guidelines PDF___FAS Guidelines 05102021_e02b6d0f98ac.pdf | 51 pages | 12782 words
  Ingested: CBIC GST Invoice Rules___Citizen Charter_cc6d0fa3a0c6.pdf | 5 pages | 983 words
  Ingested: CBIC GST Invoice Rules___Click here for Advisory_9310279e65fe.pdf | 2 pages | 269 words
  Ingested: CBIC GST Invoice Rules___Click here for_                Advisory_1a42739622f4.pdf | 2 pages | 437 words
  Ingested: CBIC GST Invoice Rules___Click here for_                Advisory_4df4b5d6870d.pdf | 47 pages | 2164 words
  Ingested: CBIC GST Invoice Rules___Click here for_                Advisory_711e24d25037.pdf | 37 pages | 1626 words
  Ingested: CBIC GST Invoice Rules___Click here for_                Advisory_d87c6de5c808.pdf | 52 pages | 1640 words
  Ingested: CBIC GST Invoice Rules___Click here for_                Advisory_f2b7f3f5858d.pdf | 24 pages | 975 words
  Ingested: CBIC GST Invoice Rules___Click here_                for Advisory_395328a405c5.pdf | 1 pa

In [8]:
import json, os, re, hashlib

def chunk_text(text, size=400, overlap=80):
    words = (text or '').split()
    chunks, i = [], 0
    while i < len(words):
        piece = ' '.join(words[i:i + size]).strip()
        if piece:
            chunks.append(piece)
        i += size - overlap
    return chunks

all_chunks, cid = [], 0
seen_chunk_text = set()
source_docs = 0

# Load the project SOP pack and the source-attributed public PDF pages.
for fname in ['franchise_kb.jsonl', 'pdf_extracted.jsonl']:
    fpath = f'{KB_DIR}/{fname}'
    if not os.path.exists(fpath):
        continue
    with open(fpath, encoding='utf-8') as f:
        for line in f:
            if not line.strip():
                continue
            doc = json.loads(line)
            content = doc.get('content', '')
            if len(content.split()) < 20:
                continue
            source_docs += 1
            for chunk_no, chunk in enumerate(chunk_text(content)):
                normalized = re.sub(r'\s+', ' ', chunk).strip().lower()
                digest = hashlib.sha256(normalized.encode('utf-8')).hexdigest()
                if digest in seen_chunk_text:
                    continue
                seen_chunk_text.add(digest)
                all_chunks.append({
                    'chunk_id': f'fops_{cid:06d}',
                    'title': doc.get('title', 'Untitled document'),
                    'text': chunk,
                    'source': doc.get('source') or fname.replace('.jsonl', ''),
                    'category': doc.get('category', 'Franchise operations'),
                    'chunk_no': chunk_no,
                    'words': len(chunk.split())
                })
                cid += 1

chunks_path = f'{KB_DIR}/all_chunks.jsonl'
with open(chunks_path, 'w', encoding='utf-8') as f:
    for c in all_chunks:
        f.write(json.dumps(c, ensure_ascii=False) + '\n')

pdf_urls = set(c['source'] for c in all_chunks if str(c.get('source', '')).startswith('http'))
print('Chunking complete:')
print(f'  Source/page records processed: {source_docs}')
print(f'  Unique public source URLs represented: {len(pdf_urls)}')
print(f'  Exact duplicate chunks removed: {len(seen_chunk_text)} unique chunks retained from {cid} candidates')
print(f'  Total unique chunks: {len(all_chunks):,}')
print(f'  Average chunk size: {sum(c["words"] for c in all_chunks) / max(1, len(all_chunks)):.0f} words')
print(f'  Saved to Drive: {chunks_path}')


Chunking complete:
  Source/page records processed: 11392
  Unique public source URLs represented: 471
  Exact duplicate chunks removed: 20385 unique chunks retained from 20385 candidates
  Total unique chunks: 20,385
  Average chunk size: 259 words
  Saved to Drive: /content/drive/MyDrive/FranchiseOps_AI/synthetic_kb/all_chunks.jsonl


In [9]:
import json, numpy as np, faiss, os, time
from sentence_transformers import SentenceTransformer

print('Loading sentence-transformer (cached locally)...')
t0 = time.time()
embedder = SentenceTransformer('all-MiniLM-L6-v2', cache_folder=ST_CACHE)
print(f'Model ready in {time.time()-t0:.1f}s')

with open(f'{KB_DIR}/all_chunks.jsonl') as f:
    chunks = [json.loads(l) for l in f]

texts = [c['text'] for c in chunks]
print(f'Encoding {len(texts)} chunks...')
t1 = time.time()
embeddings = embedder.encode(texts, batch_size=128, show_progress_bar=True, convert_to_numpy=True).astype('float32')
faiss.normalize_L2(embeddings)   # Cosine similarity via dot product
print(f'Encoding done in {time.time()-t1:.1f}s')

dim = embeddings.shape[1]
nlist = max(1, min(32, len(chunks) // 4))
quantizer = faiss.IndexFlatIP(dim)
index = faiss.IndexIVFFlat(quantizer, dim, nlist, faiss.METRIC_INNER_PRODUCT)
index.train(embeddings)
index.add(embeddings)
index.nprobe = 8   # Search 8 clusters - good speed/recall tradeoff

idx_path  = f'{FAISS_DIR}/franchise_faiss.index'
meta_path = f'{FAISS_DIR}/franchise_chunks_meta.json'
faiss.write_index(index, idx_path)
with open(meta_path, 'w') as f:
    json.dump(chunks, f)

print(f'FAISS index saved to Drive:')
print(f'  {idx_path} ({os.path.getsize(idx_path)/1024:.1f} KB)')
print(f'  Vectors: {index.ntotal} | Dim: {dim} | Type: IVFFlat cosine')


Loading sentence-transformer (cached locally)...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model ready in 3.9s
Encoding 20385 chunks...


Batches:   0%|          | 0/160 [00:00<?, ?it/s]

Encoding done in 3401.6s
FAISS index saved to Drive:
  /content/drive/MyDrive/FranchiseOps_AI/faiss_indexes/franchise_faiss.index (30785.1 KB)
  Vectors: 20385 | Dim: 384 | Type: IVFFlat cosine


In [10]:
import json, pickle, os
from rank_bm25 import BM25Okapi

with open(f'{KB_DIR}/all_chunks.jsonl') as f:
    chunks = [json.loads(l) for l in f]

tokenized = [c['text'].lower().split() for c in chunks]
bm25 = BM25Okapi(tokenized)

bm25_path = f'{BM25_DIR}/franchise_bm25.pkl'
with open(bm25_path, 'wb') as f:
    pickle.dump({'bm25': bm25, 'chunks': chunks}, f)

print(f'BM25 index saved to Drive:')
print(f'  {bm25_path} ({os.path.getsize(bm25_path)/1024:.1f} KB)')
print(f'  Documents indexed: {len(chunks)}')


BM25 index saved to Drive:
  /content/drive/MyDrive/FranchiseOps_AI/bm25_indexes/franchise_bm25.pkl (82193.2 KB)
  Documents indexed: 20385


In [11]:
import json, numpy as np, faiss, pickle, time
from sentence_transformers import SentenceTransformer

class FranchiseRAG:
    def __init__(self):
        t0 = time.time()
        self.index    = faiss.read_index(f'{FAISS_DIR}/franchise_faiss.index')
        self.index.nprobe = 8
        with open(f'{FAISS_DIR}/franchise_chunks_meta.json') as f:
            self.chunks = json.load(f)
        with open(f'{BM25_DIR}/franchise_bm25.pkl', 'rb') as f:
            data = pickle.load(f)
        self.bm25     = data['bm25']
        self.embedder = SentenceTransformer('all-MiniLM-L6-v2', cache_folder=ST_CACHE)
        print(f'FranchiseRAG ready in {time.time()-t0:.1f}s | {len(self.chunks)} chunks')

    def search(self, query, k=4, alpha=0.6):
        q_emb = self.embedder.encode([query], convert_to_numpy=True).astype('float32')
        faiss.normalize_L2(q_emb)
        scores, idxs = self.index.search(q_emb, k*2)
        dense = [{'chunk': self.chunks[i], 'ds': float(s)} for s, i in zip(scores[0], idxs[0]) if i >= 0]
        tokens = query.lower().split()
        bm25_s = self.bm25.get_scores(tokens)
        top_k  = np.argsort(bm25_s)[::-1][:k*2]
        sparse = [{'chunk': self.chunks[i], 'bs': float(bm25_s[i])} for i in top_k if bm25_s[i] > 0]
        scored = {}
        if dense:
            dm = max(c['ds'] for c in dense) or 1.0
            for c in dense:
                cid = c['chunk']['chunk_id']
                scored[cid] = scored.get(cid, {'chunk': c['chunk'], 'score': 0.0})
                scored[cid]['score'] += alpha * (c['ds'] / dm)
        if sparse:
            bm = max(c['bs'] for c in sparse) or 1.0
            for c in sparse:
                cid = c['chunk']['chunk_id']
                scored[cid] = scored.get(cid, {'chunk': c['chunk'], 'score': 0.0})
                scored[cid]['score'] += (1-alpha) * (c['bs'] / bm)
        ranked = sorted(scored.values(), key=lambda x: x['score'], reverse=True)[:k]
        return [{'text': r['chunk']['text'], 'title': r['chunk']['title'],
                 'source': r['chunk']['source'], 'score': r['score']} for r in ranked]

rag = FranchiseRAG()

TEST_QUERIES = [
    'What is the minimum CSAT score required?',
    'How should I handle high staff attrition?',
    'What are the Tier 1 outlet requirements?',
    'What corrective action is needed for low audit scores?',
    'How should inventory reorder be triggered?',
    'What is the recommended marketing budget allocation?',
    'What are the financial reporting requirements?',
]

print('Query | Score | Source | Latency')
print('-' * 80)
latencies = []
for q in TEST_QUERIES:
    t = time.time()
    results = rag.search(q, k=1)
    ms = (time.time()-t)*1000
    latencies.append(ms)
    r = results[0] if results else {}
    print(f'{q[:40]:42} | {r.get("score",0):.3f} | {r.get("title","")[:20]:22} | {ms:.0f}ms')
print(f'Average latency: {sum(latencies)/len(latencies):.0f}ms')


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

FranchiseRAG ready in 4.0s | 20385 chunks
Query | Score | Source | Latency
--------------------------------------------------------------------------------
What is the minimum CSAT score required?   | 0.600 | Standards Formulatio   | 110ms
How should I handle high staff attrition   | 0.600 | Can Large Language M   | 100ms
What are the Tier 1 outlet requirements?   | 0.600 | SOP-006: Outlet Tier   | 101ms
What corrective action is needed for low   | 0.600 | SOP-005: Audit and C   | 126ms
How should inventory reorder be triggere   | 1.000 | SOP-002: Inventory M   | 103ms
What is the recommended marketing budget   | 0.600 | A Unified Framework    | 97ms
What are the financial reporting require   | 0.600 | Date Extended to 14/   | 90ms
Average latency: 104ms


In [12]:
RAG_ENGINE = '''
import os, json, pickle, numpy as np

_retriever = None
FAISS_DIR = '/content/drive/MyDrive/FranchiseOps_AI/faiss_indexes'
BM25_DIR  = '/content/drive/MyDrive/FranchiseOps_AI/bm25_indexes'
ST_CACHE  = '/content/.cache/sentence_transformers'

def _load_retriever():
    global _retriever
    if _retriever is not None: return _retriever
    try:
        import faiss
        from sentence_transformers import SentenceTransformer
        idx = faiss.read_index(f'{FAISS_DIR}/franchise_faiss.index')
        idx.nprobe = 8
        with open(f'{FAISS_DIR}/franchise_chunks_meta.json') as f: chunks = json.load(f)
        with open(f'{BM25_DIR}/franchise_bm25.pkl', 'rb') as f: data = pickle.load(f)
        emb = SentenceTransformer('all-MiniLM-L6-v2', cache_folder=ST_CACHE)
        _retriever = {'index': idx, 'chunks': chunks, 'bm25': data['bm25'], 'embedder': emb}
        return _retriever
    except Exception as e:
        return None

def is_rag_ready():
    return os.path.exists(f'{FAISS_DIR}/franchise_faiss.index')

def retrieve(query, k=4, alpha=0.6):
    r = _load_retriever()
    if r is None:
        return [{'text': 'Run FranchiseOps_RAG_Builder.ipynb to build the index.', 'title': 'System', 'source': 'system', 'score': 0}]
    import faiss
    q_emb = r['embedder'].encode([query], convert_to_numpy=True).astype('float32')
    faiss.normalize_L2(q_emb)
    scores, idxs = r['index'].search(q_emb, k*2)
    dense = [{'chunk': r['chunks'][i], 'ds': float(s)} for s, i in zip(scores[0], idxs[0]) if i >= 0]
    tokens = query.lower().split()
    bm25_s = r['bm25'].get_scores(tokens)
    top_k  = np.argsort(bm25_s)[::-1][:k*2]
    sparse = [{'chunk': r['chunks'][i], 'bs': float(bm25_s[i])} for i in top_k if bm25_s[i] > 0]
    scored = {}
    if dense:
        dm = max(c['ds'] for c in dense) or 1.0
        for c in dense:
            cid = c['chunk']['chunk_id']
            scored[cid] = scored.get(cid, {'chunk': c['chunk'], 'score': 0.0})
            scored[cid]['score'] += alpha * (c['ds'] / dm)
    if sparse:
        bm = max(c['bs'] for c in sparse) or 1.0
        for c in sparse:
            cid = c['chunk']['chunk_id']
            scored[cid] = scored.get(cid, {'chunk': c['chunk'], 'score': 0.0})
            scored[cid]['score'] += (1-alpha) * (c['bs'] / bm)
    ranked = sorted(scored.values(), key=lambda x: x['score'], reverse=True)[:k]
    return [{'text': r['chunk']['text'], 'title': r['chunk']['title'], 'source': r['chunk']['source'], 'score': r['score']} for r in ranked]

def answer_with_citation(query):
    results = retrieve(query)
    if not results or results[0]['score'] == 0:
        return 'No relevant context found.', 'None'
    return ' '.join([r['text'] for r in results]), results[0]['title']
'''

with open('/content/rag_engine.py', 'w') as f:
    f.write(RAG_ENGINE)
print('Exported: /content/rag_engine.py')
print('This module auto-loads FAISS index from Drive on first query.')


Exported: /content/rag_engine.py
This module auto-loads FAISS index from Drive on first query.


In [13]:
import plotly.express as px, pandas as pd, os

QUERIES = [
    'What is the minimum CSAT score required?',
    'How should I handle high staff attrition?',
    'What are the Tier 1 outlet requirements?',
    'What corrective action is needed for audit scores below 70?',
    'How should inventory reorder be triggered?',
    'What is the recommended marketing budget allocation?',
    'What happens when operating cost exceeds 65%?',
    'What are the health and safety standards?',
]
all_results = []
for q in QUERIES:
    for r in rag.search(q, k=3):
        all_results.append({'Query': q[:35]+'..', 'Title': r['title'][:28], 'Score': r['score'], 'Source': r['source']})

df = pd.DataFrame(all_results)
fig = px.bar(df.groupby('Source')['Score'].mean().reset_index(),
    x='Source', y='Score', color='Source', title='Average Retrieval Score by Source')
fig.show()

fig2 = px.box(df, x='Source', y='Score', color='Source', title='Score Distribution by Source')
fig2.show()

print('=' * 60)
print('  FRANCHISEOPS RAG BUILDER - COMPLETE')
print('=' * 60)
files = [
    (f'{FAISS_DIR}/franchise_faiss.index', 'FAISS dense index'),
    (f'{FAISS_DIR}/franchise_chunks_meta.json', 'Chunk metadata'),
    (f'{BM25_DIR}/franchise_bm25.pkl', 'BM25 sparse index'),
    (f'{KB_DIR}/franchise_kb.jsonl', 'Synthetic knowledge base'),
    ('/content/rag_engine.py', 'RAG engine module'),
]
for path, desc in files:
    exists = os.path.exists(path)
    size = os.path.getsize(path)/1024 if exists else 0
    print(f'  {chr(9989) if exists else chr(10060)} {desc}: {path.split("/")[-1]} ({size:.1f} KB)')
print()
print('Files in Drive persist between Colab sessions!')
print('LLM weights stay local (/content/) to save Drive space.')
print(f'Avg retrieval score: {df["Score"].mean():.3f}')


  FRANCHISEOPS RAG BUILDER - COMPLETE
  ✅ FAISS dense index: franchise_faiss.index (30785.1 KB)
  ✅ Chunk metadata: franchise_chunks_meta.json (44508.8 KB)
  ✅ BM25 sparse index: franchise_bm25.pkl (82193.2 KB)
  ✅ Synthetic knowledge base: franchise_kb.jsonl (11.8 KB)
  ✅ RAG engine module: rag_engine.py (2.7 KB)

Files in Drive persist between Colab sessions!
LLM weights stay local (/content/) to save Drive space.
Avg retrieval score: 0.667


# v2 · Hybrid retrieval, evaluation and export for the app
Uses the chunks written above (`all_chunks.jsonl`). Same BM25 / tokenizer code as the app, so numbers measured here are the numbers you get there.

In [14]:
import re, json, math, base64, collections, time, os
import numpy as np, pandas as pd, torch
from sentence_transformers import SentenceTransformer, CrossEncoder
EMB_MODEL = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'   # what the app uses (multilingual: Hindi/Tamil... queries work too)
RERANK_MODEL = 'cross-encoder/ms-marco-MiniLM-L-6-v2'
_STOP = set("a an the of to in on for and or is are was were be been it this that these those with as at by from how what when which who whom do does did can could should would will may might must not no if then than so we you i they he she them their our your its into about over under per each any all more most such via".split())
def _stem(w):
    for suf in ("ations", "ation", "ingly", "ings", "ing", "edly", "ed", "ies", "es", "ly", "s"):
        if len(w) > len(suf) + 3 and w.endswith(suf): return w[: -len(suf)] + ("y" if suf == "ies" else "")
    return w
def tok(t): return [_stem(w) for w in re.findall(r"[a-z0-9]+|[^\x00-\x7f\s]+", (t or "").lower()) if w not in _STOP]

class BM25:
    """Okapi BM25 (k1=1.5, b=0.75) over pre-tokenised documents; score() returns a numpy vector over all docs."""
    def __init__(self, docs, k1=1.5, b=0.75):
        self.k1, self.b = k1, b; self.n = len(docs); self.tf = [collections.Counter(d) for d in docs]; self.len = np.array([len(d) for d in docs], dtype="float32"); self.avg = float(self.len.mean()) if self.n else 1.0
        df = collections.Counter(); self.post = collections.defaultdict(list)
        for i, c in enumerate(self.tf):
            for w, f in c.items(): df[w] += 1; self.post[w].append((i, f))
        self.idf = {w: math.log(1 + (self.n - d + .5) / (d + .5)) for w, d in df.items()}
    def score(self, q):
        s = np.zeros(self.n, dtype="float32")
        for w in set(q):
            if w not in self.idf: continue
            idf = self.idf[w]
            for i, f in self.post[w]: s[i] += idf * f * (self.k1 + 1) / (f + self.k1 * (1 - self.b + self.b * self.len[i] / self.avg))
        return s



In [15]:
rows = [json.loads(l) for l in open(f'{KB_DIR}/all_chunks.jsonl')]
rows = [dict(title=r['title'], source=r.get('source','kb'), text=r['text']) for r in rows if len(r['text']) > 20]
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
emb = SentenceTransformer(EMB_MODEL, device=dev); rer = CrossEncoder(RERANK_MODEL, device=dev, max_length=384)
t = time.time(); E = emb.encode([r['text'] for r in rows], batch_size=64, normalize_embeddings=True, show_progress_bar=False).astype('float32'); print(len(rows), 'chunks embedded in', round(time.time()-t,1), 's on', dev)
bm = BM25([tok(r['title'] + ' ' + r['text']) for r in rows])

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

... (12 outputs left out)

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

20304 chunks embedded in 2955.2 s on cpu


In [16]:
def rank(v, n=30): return [int(i) for i in np.argsort(-v)[:n] if v[i] > 0]
def search(q, k=5, mode='hybrid', rerank=True):
    b = bm.score(tok(q)); d = E @ emb.encode([q], normalize_embeddings=True)[0]; lists = []
    if mode in ('bm25', 'hybrid'): lists.append(rank(b))
    if mode in ('dense', 'hybrid'): lists.append(rank(d + 1))
    rrf = collections.defaultdict(float)
    for l in lists:
        for r, i in enumerate(l): rrf[i] += 1 / (60 + r + 1)
    order = sorted(rrf, key=lambda i: -rrf[i])[:12]
    if rerank and mode != 'bm25' and order:
        s = rer.predict([(q, rows[i]['text']) for i in order], show_progress_bar=False); order = [order[j] for j in np.argsort(-s)]
    return [rows[i] for i in order[:k]]
QS = ['What is the minimum CSAT score required?', 'How should I handle high staff attrition?', 'What corrective action is needed for audit scores below 70?', 'How should inventory reorder be triggered?']
for q in QS:
    print('\nQ:', q)
    for m, rr in (('bm25', False), ('hybrid', True)):
        print('  ', m.ljust(7), '->', [h['title'][:34] for h in search(q, 3, m, rr)])


Q: What is the minimum CSAT score required?
   bm25    -> ['SOP-001: Customer Service Standard', 'BP-004: Customer Experience Excell', 'KPI-003: Customer Metrics Benchmar']
   hybrid  -> ['SOP-001: Customer Service Standard', 'BP-004: Customer Experience Excell', 'KPI-003: Customer Metrics Benchmar']

Q: How should I handle high staff attrition?
   bm25    -> ['SOP-003: Staff Attrition Managemen', 'MitigatingAttrition: Data-Driven A', 'BP-002: Staff Retention Playbook']
   hybrid  -> ['SOP-003: Staff Attrition Managemen', 'Proactive Intervention to Downtren', 'Can Large Language ModelPredictEmp']

Q: What corrective action is needed for audit scores below 70?
   bm25    -> ['SOP-005: Audit and Compliance Fram', 'Inviting stakeholders’ comments on', 'Lab Recognition Scheme 2020 - page']
   hybrid  -> ['SOP-005: Audit and Compliance Fram', 'Lab Recognition Scheme 2020 - page', 'Lab Recognition Scheme 2020 - page']

Q: How should inventory reorder be triggered?
   bm25    -> ['SOP-002: I

In [17]:
# Evaluate on your own labelled questions: a qa.csv with columns  question,title  (title = the document that should come back).
import os
qa = pd.read_csv('qa.csv') if os.path.exists('qa.csv') else None
if qa is None: print('No qa.csv found: upload one (question,title) to get recall@1 / recall@3 / MRR for each method.')
else:
    out = []
    for name, mode, rr in (('BM25 only', 'bm25', False), ('Dense only', 'dense', False), ('Hybrid (RRF)', 'hybrid', False), ('Hybrid + rerank', 'hybrid', True)):
        r1 = r3 = mrr = 0
        for q, want in zip(qa.question, qa.title):
            seen = []
            for h in search(q, 8, mode, rr):
                if h['title'] not in seen: seen.append(h['title'])
            if want in seen: p = seen.index(want) + 1; r1 += p == 1; r3 += p <= 3; mrr += 1 / p
        n = len(qa); out.append(dict(method=name, recall_at_1=round(r1/n,2), recall_at_3=round(r3/n,2), MRR=round(mrr/n,2)))
    display(pd.DataFrame(out))

No qa.csv found: upload one (question,title) to get recall@1 / recall@3 / MRR for each method.


In [18]:
# Export for the app (embeddings included). Saved to Drive and offered as a download.
export = dict(format='kb-v1', emb_model=EMB_MODEL, chunks=[dict(title=r['title'], source=r['source'], text=r['text'], emb=base64.b64encode(E[i].tobytes()).decode()) for i, r in enumerate(rows)])
out = f'{DRIVE_BASE}/kb_export.json'; json.dump(export, open(out, 'w')); print(round(os.path.getsize(out)/1024), 'KB ->', out)
print('Drive corpus:', f'{KB_DIR}/all_chunks.jsonl')
print('Source manifests:', f'{DRIVE_BASE}/manifest.json', f'{DRIVE_BASE}/pdf_document_catalog.csv')
try:
    from google.colab import files; files.download(out)
except Exception: pass

83669 KB -> /content/drive/MyDrive/FranchiseOps_AI/kb_export.json
Drive corpus: /content/drive/MyDrive/FranchiseOps_AI/synthetic_kb/all_chunks.jsonl
Source manifests: /content/drive/MyDrive/FranchiseOps_AI/manifest.json /content/drive/MyDrive/FranchiseOps_AI/pdf_document_catalog.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Milestone 3 Task 1 — load the corpus into the final app

After the builder finishes, open the final FranchiseOps app notebook with `USE_DRIVE = True` and `PERSIST = True`, start the app, then sign in as Admin.

1. Open **AI Copilot → Knowledge base**.
2. Choose **Import from Drive**. This calls the app's `/api/kb/drive` endpoint and refreshes its `builtin=2` knowledge chunks from `MyDrive/FranchiseOps_AI/synthetic_kb/all_chunks.jsonl`.
3. Open **Admin → System check** and confirm both **Knowledge base (RAG)** and **Chunks imported from Drive** are healthy and non-zero.
4. Ask 10 knowledge questions across food safety/FSSAI, labour, franchise agreements, customer service, stock/supply chain, marketing/ROI, audit, and GST/finance. Check that answers include source citations.

**Important:** the final app may retain previously imported Drive chunks in its persistent SQLite database. After rebuilding the corpus, click **Import from Drive** again so the new corpus replaces the older Drive-imported rows.
